In [ ]:
import numpy as np
from sympy import *

import torch
from torch import nn
# import torch.nn.functional as F
# import torch.optim as optim
# import torch.nn.init as init

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

In [16]:
# t, delta, omega = symbols("t delta omega")
t = Symbol("t", positive=True)

m = 1
mu = 5
k = 500

delta = mu/(2*m)
w0 = sqrt(k)
w = sqrt(w0**2 - delta**2)

phi = tan(-delta/w)**(-1)
A = 1/(2 * cos(phi))

display(A)

x = exp(-delta*t)*(2*A*cos(phi + w*t))
display(x.simplify())

-0.595470930409999

-1.19094186082*exp(-2.5*t)*cos(22.220486043289*t - 8.85065980104099)

In [34]:
times = np.random.rand(10) * 0.5
D = np.array([x.subs(t, time) for time in times])
D

array([-0.240123049628834, 0.384779473160838, -0.440846785951372,
       -0.744084194897123, -0.887869163926613, 0.873451830717956,
       -0.159600248631267, -0.571276878807794, -0.739070355075165,
       0.539313565603856], dtype=object)

In [37]:
simple_model = nn.Sequential(
    # input layer with 1 input
    nn.Linear(1, 10),
    nn.Tanh(),
    # hidden layer
    nn.Linear(10, 1)
)

print(simple_model)

Sequential(
  (0): Linear(in_features=1, out_features=10, bias=True)
  (1): Tanh()
  (2): Linear(in_features=10, out_features=1, bias=True)
)


In [ ]:
optimizer = optim.SGD(net.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
# we could have done this ourselves,
# but we should be aware of sklearn and its tools
from sklearn.metrics import accuracy_score

# setting hyperparameters and gettings epoch sizes
batch_size = 100
num_epochs = 200
num_samples_train = x_train.shape[0]
num_batches_train = num_samples_train // batch_size
num_samples_valid = x_valid.shape[0]
num_batches_valid = num_samples_valid // batch_size

# setting up lists for handling loss/accuracy
train_acc, train_loss = [], []
valid_acc, valid_loss = [], []
test_acc, test_loss = [], []
cur_loss = 0
losses = []

get_slice = lambda i, size: range(i * size, (i + 1) * size)

for epoch in range(num_epochs):
    # Forward -> Backprob -> Update params
    ## Train
    cur_loss = 0
    net.train()
    for i in range(num_batches_train):
        optimizer.zero_grad()
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        # compute gradients given loss
        target_batch = targets_train[slce]
        batch_loss = criterion(output, target_batch)
        batch_loss.backward()
        optimizer.step()
        
        cur_loss += batch_loss   
    losses.append(cur_loss / batch_size)

    net.eval()
    ### Evaluate training
    train_preds, train_targs = [], []
    for i in range(num_batches_train):
        slce = get_slice(i, batch_size)
        output = net(x_train[slce])
        
        preds = torch.max(output, 1)[1]
        
        train_targs += list(targets_train[slce].numpy())
        train_preds += list(preds.data.numpy())
    
    ### Evaluate validation
    val_preds, val_targs = [], []
    for i in range(num_batches_valid):
        slce = get_slice(i, batch_size)